In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

CFG = yaml.safe_load(open("../src/config.yaml"))
RAW_DIR   = os.path.join("..", CFG["paths"]["raw_dir"])
CLEAN_DIR = os.path.join("..", CFG["paths"]["clean_dir"])
FIGS_DIR  = os.path.join("..", CFG["paths"]["figs_dir"])
TABS_DIR  = os.path.join("..", CFG["paths"]["tables_dir"])
LOGS_DIR  = os.path.join("..", CFG["paths"]["logs_dir"])

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)
os.makedirs(TABS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
np.random.seed(CFG["policies"]["seed"])

In [2]:
def assemble_timestamp(df, dayfirst=True):
    """
    Returns a pandas.DatetimeIndex/Series named 'timestamp' built from the best available columns.
    Tries (in order):
      1) Any single column that already looks like a full timestamp.
      2) Common "date"+"time" combinations (e.g., 'Read Date' + 'End Time', 'Date' + 'Time').
      3) Columns like 'Interval End', 'End Time', 'Read Date and End Time'.
    """
    cols = list(df.columns)
    lower = {c.lower(): c for c in cols}

    # 1) Single full timestamp column candidates
    single_candidates = [c for c in cols if any(k in c.lower() for k in
        ["timestamp", "date time", "datetime", "read date and end time", "interval end time"])]
    for c in single_candidates:
        ts = pd.to_datetime(df[c], dayfirst=dayfirst, errors="coerce")
        if ts.notna().mean() > 0.9:
            return ts

    # 2) Common (date + time) pairs
    date_candidates = [c for c in cols if any(k in c.lower() for k in ["read date","date"])]
    time_candidates = [c for c in cols if any(k in c.lower() for k in ["end time","time","interval end","end"])]
    # Try all pairings until one parses well
    for dc in date_candidates:
        for tc in time_candidates:
            # skip if they are the same column
            if dc == tc: 
                continue
            ts = pd.to_datetime(
                (df[dc].astype("string").str.strip() + " " + df[tc].astype("string").str.strip()),
                dayfirst=dayfirst, errors="coerce"
            )
            if ts.notna().mean() > 0.9:
                return ts

    # 3) Columns that may contain end-of-interval timestamps alone
    end_like = [c for c in cols if any(k in c.lower() for k in ["interval end","end time","read date and end time"])]
    for c in end_like:
        ts = pd.to_datetime(df[c], dayfirst=dayfirst, errors="coerce")
        if ts.notna().mean() > 0.9:
            return ts

    # Final fallback: try ANY column; pick the one that parses best
    best_col, best_score, best_ts = None, 0, None
    for c in cols:
        ts = pd.to_datetime(df[c], dayfirst=dayfirst, errors="coerce")
        score = ts.notna().mean()
        if score > best_score:
            best_col, best_score, best_ts = c, score, ts
    if best_ts is not None and best_score > 0.5:
        # Warn but return best effort
        print(f"WARNING: using best-guess timestamp column: {best_col} (parsed {best_score:.0%} of rows)")
        return best_ts

    # If we got here, we truly couldn't parse
    raise ValueError("Could not assemble timestamp: no suitable time/date columns found.")


In [1]:
from pathlib import Path

print("Config files:", CFG["data"]["files"])
for fname in CFG["data"]["files"]:
    p = Path(RAW_DIR) / fname
    print("\n---", fname, "---")
    if not p.exists():
        print("MISSING:", p)
        continue
    tmp = read_esb_csv(p, encoding=CFG["data"]["encoding"], delimiter=CFG["data"]["delimiter"])
    print("Columns:", list(tmp.columns))
    # Show first 5 rows of any columns that look like time or date
    cand = [c for c in tmp.columns if any(k in c.lower() for k in [
        "timestamp","time","date","interval","end","start","read date","read time"
    ])]
    print("Candidate time cols:", cand[:6])
    display(tmp.head(3))


NameError: name 'CFG' is not defined

In [4]:
LOCAL_TZ = CFG["data"]["timezone_local"]

# localize and convert (handle DST)
raw["timestamp_local"] = pd.to_datetime(raw["timestamp"], errors="coerce").dt.tz_localize(
    LOCAL_TZ, ambiguous="infer", nonexistent="shift_forward"
)
raw["timestamp_utc"] = raw["timestamp_local"].dt.tz_convert("UTC")
raw = raw.drop(columns=["timestamp"])  # keep the explicit local/utc columns only
raw = raw.dropna(subset=["timestamp_utc"])
raw.head(3)

NameError: name 'raw' is not defined

In [5]:
def normalize_energy(df, read_type_col_guess=("Read Type","ReadType","Type","Register","Measure"),
                     value_col_guess=("Read Value","ReadValue","Value","kWh","kW","Consumption","Active Import (kWh)"),
                     kw_to_kwh_factor=0.5):
    cols = {c.lower(): c for c in df.columns}
    # try to find a direct numeric energy column first (kWh-like)
    direct_kwh_cols = [c for c in df.columns if "kwh" in c.lower() or "consumption" in c.lower()]
    for c in direct_kwh_cols:
        v = pd.to_numeric(df[c], errors="coerce")
        if v.notna().mean() > 0.5:
            return v.rename("kWh")

    # else pick a generic value column and possibly use Read Type to convert
    val_col = next((cols[c.lower()] for c in value_col_guess if c.lower() in cols), None)
    if val_col is None:
        raise ValueError("Could not find an energy/value column (tried 'Read Value', 'kWh', 'kW', 'Consumption').")

    val = pd.to_numeric(df[val_col], errors="coerce")
    rt_col = next((cols[c.lower()] for c in read_type_col_guess if c.lower() in cols), None)

    if rt_col is None:
        # assume kWh if no type column
        return val.rename("kWh")

    rt = df[rt_col].astype("string").str.strip().str.upper()
    is_kw  = rt.str.contains("|".join([t.upper() for t in CFG["policies"]["treat_as_kw_if"]]), na=False)
    is_kwh = rt.str.contains("|".join([t.upper() for t in CFG["policies"]["treat_as_kwh_if"]]), na=False)
    kwh = np.where(is_kwh, val, np.where(is_kw, val*kw_to_kwh_factor, np.nan))
    return pd.Series(kwh, name="kWh")


In [ ]:
INTERVAL = f'{CFG["data"]["interval_minutes"]}min'

df = raw.set_index("timestamp_utc").sort_index()
s = df["kWh"].resample(INTERVAL).sum(min_count=1)   # one record/bin expected
s = s.ffill(limit=CFG["policies"]["gap_fill_limit_intervals"])  # only tiny gaps

# rebuild frame and carry local time for presentation
out = pd.DataFrame({"kWh": s})
out["timestamp_local"] = out.index.tz_convert(LOCAL_TZ)
df = out
df.head(3)

In [ ]:
def daily_quality_flags(df, daily_min=44):
    g = df["kWh"].groupby(df["timestamp_local"].dt.date)
    counts = g.count()
    flags = counts < daily_min
    return flags

# negatives -> NaN + flag column
neg_mask = df["kWh"] < 0
df.loc[neg_mask, "kWh"] = np.nan
df["flag_negative"] = neg_mask

flags = daily_quality_flags(df, daily_min=CFG["policies"]["daily_completeness_min"])
pct_incomplete = 100*flags.mean()
print(f"Incomplete days: {pct_incomplete:.1f}%")

In [ ]:
tl = df["timestamp_local"]
df["date"] = tl.dt.date.astype("string")
df["month"] = tl.dt.to_period("M").astype("string")
df["dow"] = tl.dt.day_name()
df["is_weekend"] = tl.dt.weekday >= 5

clean_path = os.path.join(CLEAN_DIR, "clean_2023_2025_30min.csv")
df.to_csv(clean_path, index_label="timestamp_utc")
clean_path

In [ ]:
# exclude incomplete days
valid_days = pd.to_datetime(flags.index[~flags]).date
df["day"] = df["timestamp_local"].dt.normalize().dt.date
dfv = df[df["day"].isin(valid_days)].dropna(subset=["kWh"]).copy()

dfv["slot"] = dfv["timestamp_local"].dt.strftime("%H:%M")
slots = [f"{h:02d}:{m:02d}" for h in range(24) for m in (0,30)]
typical = dfv.groupby("slot")["kWh"].mean().reindex(slots)

typ_csv = os.path.join(TABS_DIR, "typical_day.csv")
typical.to_csv(typ_csv)

plt.figure()
plt.plot(typical.index, typical.values)
plt.title("Typical Day Profile (Average kWh per 30-min)")
plt.xlabel("Time of Day (Europe/Dublin)")
plt.ylabel("kWh / 30-min")
plt.xticks(rotation=45); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, "typical_day_2023_2025.png"),
            dpi=CFG["policies"]["export_dpi"])
plt.close()

typ_csv

In [ ]:
wk = dfv[~dfv["is_weekend"]].groupby("slot")["kWh"].mean().reindex(slots)
we = dfv[ dfv["is_weekend"]].groupby("slot")["kWh"].mean().reindex(slots)
wkwd = pd.DataFrame({"Weekday": wk, "Weekend": we})
wkwd.to_csv(os.path.join(TABS_DIR, "weekday_weekend_profile.csv"))

plt.figure()
plt.plot(wkwd.index, wkwd["Weekday"].values, label="Weekday")
plt.plot(wkwd.index, wkwd["Weekend"].values, label="Weekend")
plt.title("Weekday vs Weekend (Average kWh per 30-min)")
plt.xlabel("Time of Day (Europe/Dublin)")
plt.ylabel("kWh / 30-min")
plt.legend(); plt.xticks(rotation=45); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, "weekday_weekend_overlay.png"),
            dpi=CFG["policies"]["export_dpi"])
plt.close()

In [ ]:
monthly = df.groupby(df["timestamp_local"].dt.to_period("M"))["kWh"].sum().to_timestamp()
monthly.to_csv(os.path.join(TABS_DIR, "monthly_totals.csv"))

plt.figure()
plt.bar(monthly.index.strftime("%Y-%m"), monthly.values)
plt.title("Monthly Energy Use (kWh)")
plt.xlabel("Month")
plt.ylabel("kWh")
plt.xticks(rotation=45); plt.grid(True, axis="y"); plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, "monthly_totals.png"),
            dpi=CFG["policies"]["export_dpi"])
plt.close()

season = np.where(df["timestamp_local"].dt.month.isin([10,11,12,1,2,3,4]),
                  "Heating (Oct–Apr)", "Non-Heating (May–Sep)")
season_totals = df.groupby(season)["kWh"].sum()
season_totals.to_csv(os.path.join(TABS_DIR, "season_totals.csv"))
season_totals

In [ ]:
# Annual totals
annual = df.groupby(df["timestamp_local"].dt.year)["kWh"].sum()

# Avg daily per year
avg_daily_per_year = []
for y in annual.index:
    sub = df[df["timestamp_local"].dt.year==y]
    daily = sub.groupby(sub["timestamp_local"].dt.date)["kWh"].sum()
    avg_daily_per_year.append(daily.mean())

# Peak slot & base load from typical day
peak_slot = typical.idxmax()
peak_kwh  = typical.max()
night_slots = [f"{h:02d}:{m:02d}" for h in range(0,6) for m in (0,30)]
base_load = typical.loc[typical.index.isin(night_slots)].median()

# Weekend share
wknd_total = df[df["is_weekend"]]["kWh"].sum()
all_total  = df["kWh"].sum()
weekend_share = (wknd_total / all_total)*100 if all_total>0 else np.nan

kpi = pd.DataFrame({
    "year": annual.index,
    "annual_kWh": annual.values,
    "avg_daily_kWh": avg_daily_per_year
})
kpi["peak_slot"] = peak_slot
kpi["peak_slot_avg_kWh"] = peak_kwh
kpi["base_load_kWh_per_30min"] = base_load
kpi["weekend_share_percent"] = weekend_share

kpi_path = os.path.join(TABS_DIR, "kpi_table.csv")
kpi.to_csv(kpi_path, index=False)
kpi_path

In [1]:
from datetime import datetime

assumptions = f"""# Profile Assumptions

- Timezone policy: Internal processing UTC; presentation Europe/Dublin.
- DST handling: tz_localize(ambiguous='infer', nonexistent='shift_forward'), then tz_convert('UTC').
- Interval: {CFG["data"]["interval_minutes"]} minutes; resampled strictly.
- Missing data: ffill limit={CFG["policies"]["gap_fill_limit_intervals"]} intervals; longer gaps remain NaN and days may be marked incomplete.
- Units: If type=kW -> kWh via × {CFG["policies"]["kw_to_kwh_factor"]} h; prefer kWh downstream.
- Negatives: set to NaN and flagged.
- PII: identifiers dropped if found.
- Dedupe: keep='{CFG["policies"]["dedupe_keep"]}' on identical UTC timestamps.
"""

with open(os.path.join(LOGS_DIR, "profile_assumptions.md"), "w", encoding="utf-8") as f:
    f.write(assumptions)

with open(os.path.join(LOGS_DIR, "log.txt"), "a", encoding="utf-8") as f:
    f.write(f"[{datetime.now().isoformat()}] C2 run complete. Clean CSV saved, profiles & KPIs exported.\n")

"Assumptions and log updated."


NameError: name 'CFG' is not defined